In [2]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

2026-08-13 10:06:35 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Evolución vinculación Adquirencia 1 a 1

In [3]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_hist_vinc_adqui_1_a_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_hist_vinc_adqui_1_a_1 STORED AS PARQUET AS WITH news AS
  (SELECT periodo,
          num_vinc AS num_vinc_new,
          cast(left(cast(periodo AS STRING), 4) AS int) AS YEAR,
          cast(right(cast(periodo AS STRING), 2) AS int) AS mes,
          1 AS secuencia
   FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
   WHERE tipo_cliente = 'nuevos' ),
                                                                             news_out AS
  (SELECT periodo,
          num_vinc_new,
          sum(num_vinc_new) OVER (PARTITION BY YEAR
                                  ORDER BY YEAR, mes) AS num_vinc_new_cumsum_ym,
          sum(num_vinc_new) OVER (PARTITION BY secuencia
                                  ORDER BY periodo) AS num_vinc_new_cumsum
   FROM news),
                                                                             olds AS
  (SELECT periodo,
          num_vinc AS num_vinc_old
   FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
   WHERE tipo_cliente = 'viejos' ),
                                                                             alls AS
  (SELECT periodo,
          num_vinc AS num_vinc_all
   FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
   WHERE tipo_cliente = 'todos' )
SELECT n.periodo,
       n.num_vinc_new,
       n.num_vinc_new_cumsum_ym,
       n.num_vinc_new_cumsum,
       o.num_vinc_old,
       a.num_vinc_all
FROM news_out AS n
LEFT JOIN olds AS o ON n.periodo = o.periodo
LEFT JOIN alls AS a ON n.periodo = a.periodo
ORDER BY periodo DESC
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_hist_vinc_adqui_1_a_1;"""
helper.ejecutar_consulta(sql_compute)

2026-08-13 10:06:40 - [INFO] - Transcurrido: 1786633601, Tiempo de Refresco = 1000


-----------------------------------------------------------------------------------
  i  tipo              nombre                  estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------
 1/1 DROP proceso.mdo_hist_vinc_adqui_1_a_1   finalizado   10:06:40 AM     00:00.2 
-----------------------------------------------------------------------------------
-------------------------------------------------------------------------------------
  i   tipo               nombre                  estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------
 2/2 CREATE proceso.mdo_hist_vinc_adqui_1_a_1   finalizado   10:06:41 AM     00:25.8 
-------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------
  i   tipo                nombre                  estado     ho

# Uso de vinculados adquirencia 1 a 1

In [4]:
sql_drop = """DROP TABLE IF EXISTS proceso.mdo_hist_vinc_uso_adqui_1_a_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_hist_vinc_uso_adqui_1_a_1 STORED AS PARQUET AS WITH news AS
  (SELECT periodo,
          count(*) AS num_vinc_new_uso_cumsum_ym
   FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'nuevos'
   GROUP BY 1),
                                                                                 olds AS
  (SELECT periodo,
          count(*) AS num_vinc_old_uso_cumsum_ym
   FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'viejos'
   GROUP BY 1),
                                                                                 alls AS
  (SELECT periodo,
          count(*) AS num_vinc_all_uso_cumsum_ym
   FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
   WHERE tipo_cliente = 'todos'
   GROUP BY 1)
SELECT n.periodo,
       n.num_vinc_new_uso_cumsum_ym,
       o.num_vinc_old_uso_cumsum_ym,
       a.num_vinc_all_uso_cumsum_ym
FROM news AS n
LEFT JOIN olds AS o ON n.periodo = o.periodo
LEFT JOIN alls AS a ON n.periodo = a.periodo
ORDER BY n.periodo DESC;
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_hist_vinc_uso_adqui_1_a_1;"""
helper.ejecutar_consulta(sql_compute)

------------------------------------------------------------------------------------------
  i   tipo                  nombre                    estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 4/4    DROP proceso.mdo_hist_vinc_uso_adqui_1_a_1   finalizado   10:07:58 AM     00:00.1 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------
  i   tipo                  nombre                    estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 5/5  CREATE proceso.mdo_hist_vinc_uso_adqui_1_a_1   finalizado   10:07:58 AM     02:18.1 
------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------

# Tabla resultado

In [28]:
# sql = """
# WITH outcome1 AS
#   (SELECT a.fecha_ym,
#           a.num_vinc_new,
#           a.num_vinc_cumsum,
#           a.num_vinc_new_cumsum_ym,
#           nvl(b.num_vinc_new_uso_cumsum_ym, 0) AS num_vinc_new_uso_cumsum_ym,
#           round(nvl(b.num_vinc_new_uso_cumsum_ym, 0)/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
#           nvl(c.num_vinc_old_uso_cumsum_ym, 0) AS num_vinc_old_uso_cumsum_ym,
#           nvl(d.num_vinc_all_uso_cumsum_ym, 0) AS num_vinc_all_uso_cumsum_ym,
#           round(nvl(d.num_vinc_all_uso_cumsum_ym, 0)/a.num_vinc_cumsum, 4) AS num_vinc_all_prop_uso,
#           left(cast(a.fecha_ym AS string), 4) AS YEAR,
#           right(cast(a.fecha_ym AS string), 2) AS mes
#    FROM proceso.mdo_aceptacion_comercios_vinc AS a
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_new_uso AS b ON a.fecha_ym = b.fecha_ym
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_old_uso AS c ON a.fecha_ym = c.fecha_ym
#    LEFT JOIN proceso.mdo_aceptacion_comercios_num_vinc_all_uso AS d ON a.fecha_ym = d.fecha_ym),
#      outcome2 AS
#   (SELECT fecha_ym,
#           num_vinc_cumsum AS num_vinc_old,
#           cast(cast(YEAR AS int) + 1 AS string) AS YEAR
#    FROM outcome1
#    WHERE mes = '12')
# SELECT a.fecha_ym,
#        CONCAT(a.YEAR, '/', a.mes, '/', '01') AS fecha_ym2,
#        a.num_vinc_new,
#        a.num_vinc_new_cumsum_ym,
#        a.num_vinc_new_uso_cumsum_ym,
#        a.num_vinc_new_prop_uso,
#        b.num_vinc_old,
#        a.num_vinc_old_uso_cumsum_ym,
#        round(a.num_vinc_old_uso_cumsum_ym/b.num_vinc_old, 4) AS num_vinc_old_prop_uso,
#        a.num_vinc_cumsum,
#        a.num_vinc_all_uso_cumsum_ym,
#        a.num_vinc_all_prop_uso
# FROM outcome1 AS a
# LEFT JOIN outcome2 AS b ON a.year = b.year
# WHERE a.fecha_ym BETWEEN 202201 AND 202511
# ORDER BY a.fecha_ym DESC;
# """
# # print(sql)
# df_outcome = helper.obtener_dataframe(sql)
# df_outcome

In [5]:
sql = """
SELECT a.periodo,
       concat(cast(a.periodo as string), '01') AS fecha_ymd2,
       a.num_vinc_new,
       a.num_vinc_new_cumsum_ym,
       a.num_vinc_new_cumsum,
       a.num_vinc_old,
       a.num_vinc_all,
       b.num_vinc_new_uso_cumsum_ym,
       b.num_vinc_old_uso_cumsum_ym,
       b.num_vinc_all_uso_cumsum_ym,
       round(b.num_vinc_new_uso_cumsum_ym/a.num_vinc_new_cumsum_ym, 4) AS num_vinc_new_prop_uso,
       round(b.num_vinc_old_uso_cumsum_ym/a.num_vinc_old, 4) AS num_vinc_old_prop_uso,
       round(b.num_vinc_all_uso_cumsum_ym/a.num_vinc_all, 4) AS num_vinc_all_prop_uso
FROM proceso.mdo_hist_vinc_adqui_1_a_1 AS a
LEFT JOIN proceso.mdo_hist_vinc_uso_adqui_1_a_1 AS b ON a.periodo = b.periodo
ORDER BY a.periodo DESC;
"""
df_outcome = helper.obtener_dataframe(sql)
df_outcome

--------------------------------------------------------------------------------------------
  i    tipo                   nombre                    estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 7/7 DATAFRAME                                         ejecutando   10:12:28 AM             

2026-08-13 10:13:00 - [INFO] - 55 filas, 13 columnas, 00:31.6 consultando, 00:00.1 descargando, 00:00.0 convirtiendo


 7/7 DATAFRAME                                         finalizado   10:12:28 AM     00:31.8 
--------------------------------------------------------------------------------------------


,periodo,fecha_ymd2,num_vinc_new,num_vinc_new_cumsum_ym,num_vinc_new_cumsum,num_vinc_old,num_vinc_all,num_vinc_new_uso_cumsum_ym,num_vinc_old_uso_cumsum_ym,num_vinc_all_uso_cumsum_ym,num_vinc_new_prop_uso,num_vinc_old_prop_uso,num_vinc_all_prop_uso
0,202607.0,20260701,3812,23778,205509,447611,471385,12093,109345,121439,0.5086,0.2443,0.2576
1,202606.0,20260601,3107,19966,201697,447619,467583,10280,108755,119036,0.5149,0.2430,0.2546
2,202605.0,20260501,3364,16859,198590,448057,464916,8447,108073,116521,0.5010,0.2412,0.2506
3,202604.0,20260401,3739,13495,195226,448056,461551,6523,107059,113583,0.4834,0.2389,0.2461
4,202603.0,20260301,3887,9756,191487,448074,457831,4602,105678,110281,0.4717,0.2358,0.2409
5,202602.0,20260201,3200,5869,187600,448099,453969,2617,103470,106088,0.4459,0.2309,0.2337
6,202601.0,20260101,2669,2669,184400,448103,450772,1004,98960,99964,0.3762,0.2208,0.2218
7,202512.0,20251201,3412,38305,181731,409803,448104,24455,113080,137535,0.6384,0.2759,0.3069
8,202511.0,20251101,3284,34893,178319,409832,444725,22119,112697,134816,0.6339,0.2750,0.3031
9,202510.0,20251001,3759,31609,175035,409896,441505,20079,112384,132463,0.6352,0.2742,0.3000


In [6]:
df_outcome.to_excel('main_data/evolucion_vinculacion_y_uso_adquirencia_1_a_1.xlsx', index=False)

# Eliminación tablas proceso.

In [7]:
# Eliminación tablas proceso.
tablas_borrar = ['proceso.mdo_hist_vinc_adqui_1_a_1', 'proceso.mdo_hist_vinc_uso_adqui_1_a_1']

for tabla in tablas_borrar:
    sql_drop = f"""DROP TABLE IF EXISTS {tabla} PURGE;"""
    helper.ejecutar_consulta(sql_drop)

--------------------------------------------------------------------------------------------
  i    tipo                   nombre                    estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 8/8      DROP     proceso.mdo_hist_vinc_adqui_1_a_1   finalizado   10:13:53 AM     00:00.2 
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
  i    tipo                   nombre                    estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 9/9      DROP proceso.mdo_hist_vinc_uso_adqui_1_a_1   finalizado   10:13:53 AM     00:00.3 
--------------------------------------------------------------------------------------------
